In [1]:
"""
SEED 3-Class Emotion Recognition -- Ablation 2: No MSC-TimesNet (45 Sessions)
=============================================================================
Modifications:
  - Retains full 62-channel STMAE pretraining with ScalpPositionalEncoding.
  - Replaces MSC-TimesNet (FFT Top-3 Periodicity + 2D Inception Convolutions)
    with a standard 2-layer Temporal Transformer Encoder.
  - Evaluates on the full 45-session SEED dataset (15 subjects x 3 sessions)
    using 10-band features (5 log-PSD + 5 DE) under 15-fold LOSO (Seed = 42).
"""

import os
import re
import time
import math
import random
import copy
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# =====================================================================
# CONFIGURATION & PATHS
# =====================================================================
DATA_SEARCH_PATHS = [
    "/kaggle/input/datasets/yunzinan/seed-preprocessed",
    "/kaggle/input/seed-preprocessed",
    "."
]
CACHE_DIR = "/kaggle/working"
CACHE_FILE = os.path.join(CACHE_DIR, "seed_45sessions_de_psd.npz")
OUTPUT_DIR = "/kaggle/working/seed_45sessions_ablation_no_timesnet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CHANNELS = 62
NUM_CLASSES = 3         # 0: Negative, 1: Neutral, 2: Positive
RAW_DIM = 10            # 5 log-PSD + 5 DE
TRIALS_PER_SESSION = 15

SAMPLING_RATE = 200
WINDOW_SECONDS = 1
SAMPLES_PER_WINDOW = SAMPLING_RATE * WINDOW_SECONDS

FREQUENCY_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 14),
    "beta":  (14, 31),
    "gamma": (31, 50)
}

# ---- Stage A : STMAE ----
EMBEDDING_SIZE = 32
AUTOENCODER_HIDDEN_SIZE = 64
AUTOENCODER_HEADS = 4
AUTOENCODER_LAYERS = 3
AUTOENCODER_EPOCHS = 30
AUTOENCODER_LR = 1e-3
AUTOENCODER_BATCH_SIZE = 256
RANDOM_MASK_FRACTION = 0.40
REGION_MASK_CHANCE = 0.60

# ---- Stage B : Sequences ----
WINDOW_LENGTH = 10
STRIDE = 1

# ---- Stage C : Ablation Baseline (Vanilla Temporal Transformer) ----
CLASSIFIER_HIDDEN_SIZE = 128
CLASSIFIER_FEEDFORWARD_SIZE = 256
CLASSIFIER_HEADS = 4
CLASSIFIER_DROPOUT = 0.3

# ---- Training Dynamics ----
MAX_EPOCHS = 50
MIN_EPOCHS = 15
PATIENCE_EPOCHS = 10
LEARNING_RATE = 1e-3
BATCH_SIZE = 128
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
VALIDATION_FRACTION = 0.2
MIXUP_STRENGTH = 0.2
CHANNEL_DROPOUT_RATE = 0.1
RECALIBRATE_BATCHNORM = True

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHANNEL_NAMES = [
    "FP1", "FPZ", "FP2", "AF3", "AF4", "F7", "F5", "F3", "F1", "FZ",
    "F2", "F4", "F6", "F8", "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2",
    "FC4", "FC6", "FT8", "T7", "C5", "C3", "C1", "CZ", "C2", "C4",
    "C6", "T8", "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6",
    "TP8", "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8", "CB1", "O1", "OZ",
    "O2", "CB2",
]
assert len(CHANNEL_NAMES) == NUM_CHANNELS

REPORT = []


def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =====================================================================
# 62-CHANNEL SCALP TOPOLOGY
# =====================================================================
def build_scalp_coords():
    rows = [
        (["FP1", "FPZ", "FP2"], 0.95),
        (["AF3", "AF4"], 0.80),
        (["F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8"], 0.62),
        (["FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8"], 0.42),
        (["T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8"], 0.20),
        (["TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8"], -0.02),
        (["P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8"], -0.25),
        (["PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8"], -0.50),
        (["CB1", "O1", "OZ", "O2", "CB2"], -0.72),
    ]
    coords = {}
    for names, y in rows:
        n = len(names)
        xs = np.linspace(-1.0, 1.0, n) if n > 1 else np.array([0.0])
        for name, x in zip(names, xs):
            coords[name] = (float(x), float(y))
    return np.array([coords[c] for c in CHANNEL_NAMES], dtype=np.float32)


def build_regions():
    regions = defaultdict(list)
    coords = build_scalp_coords()
    for i, name in enumerate(CHANNEL_NAMES):
        x, y = coords[i]
        if abs(x) > 0.6 and -0.10 < y < 0.50:
            key = "temporal_left" if x < 0 else "temporal_right"
        elif y >= 0.55:
            key = "frontal"
        elif y >= 0.10:
            key = "central"
        elif y >= -0.35:
            key = "parietal"
        else:
            key = "occipital"
        regions[key].append(i)
    return {k: np.array(v, dtype=np.int64) for k, v in regions.items()}


SCALP_COORDS = build_scalp_coords()
REGIONS = build_regions()


# =====================================================================
# DATA EXTRACTION & CACHING
# =====================================================================
def compute_de_and_psd_1s(raw_trial_eeg):
    num_channels, num_samples = raw_trial_eeg.shape
    num_windows = num_samples // SAMPLES_PER_WINDOW
    if num_windows == 0:
        return None

    truncated = raw_trial_eeg[:, :num_windows * SAMPLES_PER_WINDOW]
    chunks = truncated.reshape(num_channels, num_windows, SAMPLES_PER_WINDOW)

    hann = np.hanning(SAMPLES_PER_WINDOW).astype(np.float32)
    windowed = chunks * hann[None, None, :]

    fft_vals = np.fft.rfft(windowed, n=SAMPLES_PER_WINDOW, axis=-1)
    psd_raw = (np.abs(fft_vals) ** 2) / float(SAMPLES_PER_WINDOW)

    band_psd, band_de = [], []
    for _, (f_low, f_high) in FREQUENCY_BANDS.items():
        power = psd_raw[:, :, f_low:f_high].mean(axis=-1)
        safe_power = np.maximum(power, 1e-10)
        band_psd.append(np.log(safe_power))
        band_de.append(0.5 * np.log(2.0 * np.pi * np.e * safe_power))

    psd_stacked = np.stack(band_psd, axis=-1)
    de_stacked = np.stack(band_de, axis=-1)
    features_10d = np.concatenate([psd_stacked, de_stacked], axis=-1)
    return np.transpose(features_10d, (1, 0, 2)).astype(np.float32)


def find_seed_raw_dir():
    for p in DATA_SEARCH_PATHS:
        if os.path.exists(os.path.join(p, "Preprocessed_EEG")):
            return os.path.join(p, "Preprocessed_EEG")
        if os.path.exists(os.path.join(p, "label.mat")):
            return p
    raise FileNotFoundError("Could not locate SEED Preprocessed_EEG directory.")


def load_or_extract_seed_45(cache_path=CACHE_FILE):
    if os.path.exists(cache_path):
        print(f"  [+] Found cached 10-band dataset at: {cache_path}")
        data = np.load(cache_path)
        return data["X"], data["y"], data["subs"], data["sess"], data["tri"]

    eeg_dir = find_seed_raw_dir()
    print(f"  [+] Ingesting 45 continuous EEG files from: {eeg_dir}")

    label_file = os.path.join(eeg_dir, "label.mat")
    mat_lbl = sio.loadmat(label_file)
    raw_labels = mat_lbl["label"].flatten() if "label" in mat_lbl else mat_lbl["labels"].flatten()
    trial_class_labels = (raw_labels + 1).astype(np.int64)

    pattern = re.compile(r"^(\d+)_(\d+)\.mat$")
    files_by_subject = {}
    for f in os.listdir(eeg_dir):
        m = pattern.match(f)
        if m:
            sub_id = int(m.group(1))
            date_str = m.group(2)
            files_by_subject.setdefault(sub_id, []).append((date_str, f))

    all_features, all_labels, all_subs, all_sess, all_tri = [], [], [], [], []
    t0 = time.time()

    for sub_id in sorted(files_by_subject.keys()):
        sorted_sessions = sorted(files_by_subject[sub_id], key=lambda x: x[0])
        for session_idx, (date_str, fname) in enumerate(sorted_sessions, start=1):
            fpath = os.path.join(eeg_dir, fname)
            mat = sio.loadmat(fpath)

            trial_keys = {}
            for k in mat.keys():
                t_match = re.search(r"eeg(\d+)$", k, re.IGNORECASE)
                if t_match:
                    trial_keys[int(t_match.group(1))] = k

            for t_num in range(1, TRIALS_PER_SESSION + 1):
                if t_num not in trial_keys:
                    continue
                raw_trial = mat[trial_keys[t_num]].astype(np.float32)
                feats = compute_de_and_psd_1s(raw_trial)
                if feats is None:
                    continue

                n_wins = feats.shape[0]
                all_features.append(feats)
                all_labels.append(np.full(n_wins, trial_class_labels[t_num - 1], dtype=np.int64))
                all_subs.append(np.full(n_wins, sub_id, dtype=np.int32))
                all_sess.append(np.full(n_wins, session_idx, dtype=np.int32))
                all_tri.append(np.full(n_wins, t_num, dtype=np.int32))

        print(f"    -> Subject {sub_id:02d} processed ({len(sorted_sessions)} sessions).")

    X = np.concatenate(all_features, axis=0)
    y = np.concatenate(all_labels, axis=0)
    subs = np.concatenate(all_subs, axis=0)
    sess = np.concatenate(all_sess, axis=0)
    tri = np.concatenate(all_tri, axis=0)

    print(f"  [+] Extracted {X.shape[0]} total frames in {(time.time() - t0):.1f}s. Saving cache...")
    np.savez_compressed(cache_path, X=X, y=y, subs=subs, sess=sess, tri=tri)
    return X, y, subs, sess, tri


def normalize_subject_session(X, subs, sess):
    Xn = X.copy()
    keys = subs * 100 + sess
    for k in np.unique(keys):
        m = (keys == k)
        blk = Xn[m]
        mu = blk.mean(axis=0, keepdims=True)
        sd = blk.std(axis=0, keepdims=True) + 1e-6
        Xn[m] = (blk - mu) / sd
    return Xn


def build_sequence_index(y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE):
    keys = subs * 1000000 + sess * 10000 + tri
    seqs, labs, s_sub, s_ses, s_tri = [], [], [], [], []
    order = np.argsort(keys, kind="stable")
    for k in np.unique(keys):
        rows = order[keys[order] == k]
        if rows.shape[0] < seq_len:
            continue
        for start in range(0, rows.shape[0] - seq_len + 1, stride):
            win = rows[start:start + seq_len]
            seqs.append(win)
            labs.append(y[win[0]])
            s_sub.append(subs[win[0]])
            s_ses.append(sess[win[0]])
            s_tri.append(tri[win[0]])
    return (np.asarray(seqs, dtype=np.int64), np.asarray(labs, dtype=np.int64),
            np.asarray(s_sub, dtype=np.int64), np.asarray(s_ses, dtype=np.int64),
            np.asarray(s_tri, dtype=np.int64))


# =====================================================================
# STAGE 2: PRETRAINED SPATIAL AUTOENCODER (STMAE)
# =====================================================================
class ScalpPositionalEncoding(nn.Module):
    def __init__(self, coords, d_model):
        super().__init__()
        self.register_buffer("coords", torch.tensor(coords, dtype=torch.float32))
        self.mlp = nn.Sequential(nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model))

    def forward(self, x):
        return x + self.mlp(self.coords).unsqueeze(0)


class STMAE(nn.Module):
    def __init__(self, in_feat, d_model=AUTOENCODER_HIDDEN_SIZE, latent_dim=EMBEDDING_SIZE,
                 heads=AUTOENCODER_HEADS, layers=AUTOENCODER_LAYERS):
        super().__init__()
        self.proj = nn.Linear(in_feat, d_model)
        self.pos = ScalpPositionalEncoding(SCALP_COORDS, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.mask_token, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.to_latent = nn.Linear(d_model, latent_dim)
        self.latent_norm = nn.LayerNorm(latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, d_model), nn.GELU(), nn.Linear(d_model, in_feat)
        )

    def encode(self, x, mask=None):
        h = self.proj(x)
        if mask is not None:
            h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
        h = self.pos(h)
        h = self.encoder(h)
        return self.latent_norm(self.to_latent(h))

    def forward(self, x, mask):
        z = self.encode(x, mask)
        return self.decoder(z), z


def sample_mask(batch_size, device):
    mask = torch.zeros(batch_size, NUM_CHANNELS, dtype=torch.bool, device=device)
    region_keys = list(REGIONS.keys())
    for b in range(batch_size):
        if random.random() < REGION_MASK_CHANCE:
            k = random.choice([1, 2])
            for key in random.sample(region_keys, k):
                mask[b, torch.tensor(REGIONS[key], device=device)] = True
        else:
            n = max(1, int(RANDOM_MASK_FRACTION * NUM_CHANNELS))
            idx = torch.randperm(NUM_CHANNELS, device=device)[:n]
            mask[b, idx] = True
    return mask


def pretrain_stmae(X_pt, epochs=AUTOENCODER_EPOCHS, lr=AUTOENCODER_LR, batch=AUTOENCODER_BATCH_SIZE):
    in_feat = X_pt.shape[2]
    model = STMAE(in_feat).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    n = X_pt.shape[0]
    model.train()
    for ep in range(1, epochs + 1):
        perm = torch.randperm(n)
        tot, nb = 0.0, 0
        for i in range(0, n, batch):
            xb = X_pt[perm[i:i + batch]]
            mask = sample_mask(xb.shape[0], DEVICE)
            recon, _ = model(xb, mask)
            m = mask.unsqueeze(-1).expand_as(xb)
            loss = F.mse_loss(recon[m], xb[m])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            opt.step()
            tot += loss.item()
            nb += 1
        sched.step()
        if ep == 1 or ep % 5 == 0:
            print("  STMAE ep%03d masked_recon_mse=%.5f" % (ep, tot / max(nb, 1)))
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "stmae_global.pt"))
    return model


# =====================================================================
# ABLATION 2: STANDARD TEMPORAL TRANSFORMER (NO MSC-TIMESNET)
# =====================================================================
class StandardTemporalTransformer(nn.Module):
    """Replaces MSC-TimesNet with a standard 2-layer temporal Transformer."""
    def __init__(self, in_dim, d_model=CLASSIFIER_HIDDEN_SIZE, num_classes=NUM_CLASSES,
                 dropout=CLASSIFIER_DROPOUT):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=CLASSIFIER_HEADS, dim_feedforward=CLASSIFIER_FEEDFORWARD_SIZE,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        h = self.inp(x)
        h = self.transformer(h)
        return self.head(h.mean(dim=1))


class AblationNoTimesNetModel(nn.Module):
    def __init__(self, stmae, in_feat, finetune_encoder=False):
        super().__init__()
        self.stmae = stmae
        self.finetune_encoder = finetune_encoder
        self.net = StandardTemporalTransformer(NUM_CHANNELS * EMBEDDING_SIZE, num_classes=NUM_CLASSES)
        for p in self.stmae.parameters():
            p.requires_grad = bool(finetune_encoder)

    def encode_seq(self, x):
        B, T, C, Fq = x.shape
        flat = x.reshape(B * T, C, Fq)
        with torch.no_grad():
            z = self.stmae.encode(flat)
        return z.reshape(B, T, C * EMBEDDING_SIZE)

    def forward(self, x):
        return self.net(self.encode_seq(x))


# =====================================================================
# FAST GPU UTILITIES
# =====================================================================
def balanced_weights(y):
    cnt = np.bincount(y, minlength=NUM_CLASSES).astype(np.float32)
    cnt[cnt == 0] = 1.0
    w = cnt.sum() / (NUM_CLASSES * cnt)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def lr_at(ep, base_lr, warmup_epochs=WARMUP_EPOCHS):
    warmup_epochs = max(1, int(warmup_epochs))
    if ep <= warmup_epochs:
        return base_lr * ep / warmup_epochs
    prog = (ep - warmup_epochs) / max(1, MAX_EPOCHS - warmup_epochs)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))


def grouped_split(labels, groups, frac=VALIDATION_FRACTION, seed=RANDOM_SEED):
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    tr, va = next(gss.split(np.zeros(len(labels)), labels, groups))
    return tr, va


def make_batches(n, batch, shuffle=True):
    idx = np.random.permutation(n) if shuffle else np.arange(n)
    for i in range(0, n, batch):
        yield idx[i:i + batch]


def apply_channel_dropout(xb, p=CHANNEL_DROPOUT_RATE):
    if p <= 0:
        return xb
    B = xb.shape[0]
    keep = (torch.rand(B, 1, NUM_CHANNELS, 1, device=xb.device) > p).float()
    return xb * keep


def mixup(xb, yb, alpha=MIXUP_STRENGTH):
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(xb.shape[0], device=xb.device)
    return lam * xb + (1 - lam) * xb[perm], yb, yb[perm], lam


@torch.no_grad()
def adabn_recalibrate(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    had_bn = any(isinstance(m, nn.BatchNorm2d) for m in model.modules())
    if not had_bn:
        return model
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()
            m.momentum = None
            m.train()
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        model(xb)
    model.eval()
    return model


@torch.no_grad()
def predict_probs(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    model.eval()
    out = []
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        out.append(F.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out, axis=0)


def fit_model(stmae, X_pt, seq_idx_pt, labels, groups, train_rows, in_feat, seed):
    seed_everything(seed)
    model = AblationNoTimesNetModel(stmae, in_feat, finetune_encoder=False).to(DEVICE)
    tr_loc, va_loc = grouped_split(labels[train_rows], groups[train_rows], seed=seed)

    tr_rows_pt = torch.tensor(train_rows[tr_loc], dtype=torch.long, device=DEVICE)
    va_rows_pt = torch.tensor(train_rows[va_loc], dtype=torch.long, device=DEVICE)

    w = balanced_weights(labels[train_rows[tr_loc]])
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW(model.net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, MAX_EPOCHS + 1):
        cur = lr_at(ep, LEARNING_RATE)
        for g in opt.param_groups:
            g["lr"] = cur

        model.train()
        for b_rows in make_batches(len(tr_rows_pt), BATCH_SIZE):
            b_idx = tr_rows_pt[b_rows]
            xb = X_pt[seq_idx_pt[b_idx]]
            yb = torch.tensor(labels[tr_rows_pt[b_rows].cpu().numpy()], dtype=torch.long, device=DEVICE)

            xb = apply_channel_dropout(xb)
            xb, ya, ybb, lam = mixup(xb, yb)
            logits = model(xb)
            loss = lam * crit(logits, ya) + (1 - lam) * crit(logits, ybb)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            opt.step()

        vp = predict_probs(model, X_pt, seq_idx_pt, va_rows_pt)
        vf1 = f1_score(labels[va_rows_pt.cpu().numpy()], vp.argmax(1), average="macro")
        if vf1 > best_f1:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if ep >= MIN_EPOCHS and bad >= PATIENCE_EPOCHS:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_f1


# =====================================================================
# EVALUATION & TRIAL METRICS
# =====================================================================
def trial_level_scores(probs, y, sub, ses, tri):
    keys = sub * 1000000 + ses * 10000 + tri
    yt, yp = [], []
    for k in np.unique(keys):
        m = (keys == k)
        true_label = np.bincount(y[m]).argmax()
        pred_label = probs[m].mean(axis=0).argmax()
        yt.append(true_label)
        yp.append(pred_label)

    yt, yp = np.array(yt), np.array(yp)
    t_acc = accuracy_score(yt, yp)
    t_bacc = balanced_accuracy_score(yt, yp)
    t_macro_f1 = f1_score(yt, yp, average="macro", zero_division=0)
    t_weighted_f1 = f1_score(yt, yp, average="weighted", zero_division=0)
    return t_acc, t_bacc, t_macro_f1, t_weighted_f1, len(yt)


def run_fold(fold_name, stmae, X_pt, seq_idx_pt, packs, train_rows, test_rows, in_feat):
    labels, s_sub, s_ses, s_tri = packs[1:]
    groups = s_sub * 1000000 + s_ses * 10000 + s_tri
    te_rows_pt = torch.tensor(test_rows, dtype=torch.long, device=DEVICE)

    stmae_fold = copy.deepcopy(stmae)
    model, _ = fit_model(stmae_fold, X_pt, seq_idx_pt, labels, groups, train_rows, in_feat, seed=RANDOM_SEED)

    if RECALIBRATE_BATCHNORM:
        model = adabn_recalibrate(model, X_pt, seq_idx_pt, te_rows_pt)
    probs = predict_probs(model, X_pt, seq_idx_pt, te_rows_pt)

    # Window-level Metrics
    yt = labels[test_rows]
    pred_win = probs.argmax(1)
    win_acc = accuracy_score(yt, pred_win)
    win_bacc = balanced_accuracy_score(yt, pred_win)
    win_macro_f1 = f1_score(yt, pred_win, average="macro", zero_division=0)
    win_weighted_f1 = f1_score(yt, pred_win, average="weighted", zero_division=0)

    # Trial-level Metrics
    t_acc, t_bacc, t_macro_f1, t_weighted_f1, num_trials = trial_level_scores(
        probs, yt, s_sub[test_rows], s_ses[test_rows], s_tri[test_rows]
    )

    print(
        f"  -> {fold_name:<11} | "
        f"WIN: acc={win_acc:.4f} bacc={win_bacc:.4f} macF1={win_macro_f1:.4f} wF1={win_weighted_f1:.4f} | "
        f"TRIAL: acc={t_acc:.4f} bacc={t_bacc:.4f} macF1={t_macro_f1:.4f} wF1={t_weighted_f1:.4f} "
        f"({len(test_rows)}w / {num_trials}t)"
    )

    REPORT.append(dict(
        fold=fold_name,
        win_acc=win_acc, win_bacc=win_bacc, win_macro_f1=win_macro_f1, win_weighted_f1=win_weighted_f1,
        trial_acc=t_acc, trial_bacc=t_bacc, trial_macro_f1=t_macro_f1, trial_weighted_f1=t_weighted_f1,
        num_windows=len(test_rows), num_trials=num_trials
    ))


def main():
    t0 = time.time()
    seed_everything(RANDOM_SEED)
    print("device:", DEVICE)
    print("Scalp regions:", {k: len(v) for k, v in REGIONS.items()})
    print("Task: SEED 3-Class LOSO (45 Sessions, 10-Band) | Ablation Mode: NO MSC-TIMESNET")

    print("\n[1] Ingesting/Extracting 10-Band Features (5 log-PSD + 5 DE)...")
    X_raw, y, subs, sess, tri = load_or_extract_seed_45()
    in_feat = X_raw.shape[2]
    print(f"shape={X_raw.shape}  features_per_channel={in_feat}  labels={np.bincount(y)}")
    print("subjects:", sorted(np.unique(subs).tolist()))
    print("sessions:", sorted(np.unique(sess).tolist()))

    print("\n[2] Normalizing (Per-Subject-Session Standardization)...")
    X = normalize_subject_session(X_raw, subs, sess)

    print("\n[2.5] Pushing normalized dataset to GPU VRAM...")
    X_pt = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    print("\n[3] Pretraining spatial masked autoencoder (STMAE) globally on 62 channels...")
    stmae = pretrain_stmae(X_pt)
    for p in stmae.parameters():
        p.requires_grad = False

    print("\n[4] Building sliding windows (T=10, Stride=1)...")
    packs = build_sequence_index(y, subs, sess, tri, WINDOW_LENGTH, stride=STRIDE)
    seq_idx = packs[0]
    seq_idx_pt = torch.tensor(seq_idx, dtype=torch.long, device=DEVICE)
    print(f"  {seq_idx.shape[0]} windows  labels={np.bincount(packs[1])}")

    # -----------------------------------------------------------------
    # LEAVE-ONE-SUBJECT-OUT (LOSO) EVALUATION
    # -----------------------------------------------------------------
    print("\n" + "=" * 84)
    print("EVALUATION: SEED 3-CLASS 45-SESSION LOSO (ABLATION 2: NO TIMESNET, SEED = 42)")
    print("=" * 84)

    s_sub = packs[2]
    unique_subs = np.unique(s_sub)
    print(f"  loso_subject       {len(unique_subs)} folds x 1 seed = {len(unique_subs)} fits")

    for sb in unique_subs:
        te_rows = np.where(s_sub == sb)[0]
        tr_rows = np.where(s_sub != sb)[0]
        fold_name = f"subject_{sb:02d}"
        run_fold(fold_name, stmae, X_pt, seq_idx_pt, packs, tr_rows, te_rows, in_feat)

    df = pd.DataFrame(REPORT)
    results_path = os.path.join(OUTPUT_DIR, "results_seed_45sessions_ablation2_no_timesnet_loso.csv")
    df.to_csv(results_path, index=False)
    print(f"\nsaved: {results_path}")

    print("\n" + "=" * 84)
    print("SUMMARY -- SEED 3-CLASS 45-SESSION LOSO (ABLATION 2: NO TIMESNET, Single-Seed = 42)")
    print("=" * 84)
    if not df.empty:
        summary_dict = {
            "win_acc": [df["win_acc"].mean()],
            "win_bacc": [df["win_bacc"].mean()],
            "win_macro_f1": [df["win_macro_f1"].mean()],
            "win_weighted_f1": [df["win_weighted_f1"].mean()],
            "trial_acc": [df["trial_acc"].mean()],
            "trial_bacc": [df["trial_bacc"].mean()],
            "trial_macro_f1": [df["trial_macro_f1"].mean()],
            "trial_weighted_f1": [df["trial_weighted_f1"].mean()],
            "folds": [len(df)]
        }
        summ = pd.DataFrame(summary_dict)
        print(summ.to_string(index=False, float_format=lambda v: "%.4f" % v))

    print(f"\nWall time: {(time.time() - t0) / 60.0:.1f} min")


if __name__ == "__main__":
    main()

device: cuda
Scalp regions: {'frontal': 14, 'temporal_left': 6, 'central': 10, 'temporal_right': 6, 'parietal': 14, 'occipital': 12}
Task: SEED 3-Class LOSO (45 Sessions, 10-Band) | Ablation Mode: NO MSC-TIMESNET

[1] Ingesting/Extracting 10-Band Features (5 log-PSD + 5 DE)...
  [+] Ingesting 45 continuous EEG files from: /kaggle/input/datasets/yunzinan/seed-preprocessed/Preprocessed_EEG
    -> Subject 01 processed (3 sessions).
    -> Subject 02 processed (3 sessions).
    -> Subject 03 processed (3 sessions).
    -> Subject 04 processed (3 sessions).
    -> Subject 05 processed (3 sessions).
    -> Subject 06 processed (3 sessions).
    -> Subject 07 processed (3 sessions).
    -> Subject 08 processed (3 sessions).
    -> Subject 09 processed (3 sessions).
    -> Subject 10 processed (3 sessions).
    -> Subject 11 processed (3 sessions).
    -> Subject 12 processed (3 sessions).
    -> Subject 13 processed (3 sessions).
    -> Subject 14 processed (3 sessions).
    -> Subject 15 pro